# AIRPORT Auction Simulations

Simulation notebook for airport time-slot instances using the AIRPORT heuristic.

In [ ]:
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from AIRPORT_generation import generate_airport_instance
from AIRPORT_heuristic import run_airport_heuristic

rng = np.random.default_rng(0)


In [ ]:
# Simulation parameters
num_items_values = [24, 48, 96]
max_cap_values = [4, 6, 8]
num_trials = 20

max_budget = 40

heuristic_kwargs = {
    "max_rounds": 200,
    "delta": 0.5,
    "epsilon_scale": 0.1,
    "price_floor": 0.0,
}


In [ ]:
results = []

for num_items in num_items_values:
    for max_cap in max_cap_values:
        num_players = num_items * (max_cap - 1) // 2

        for trial in range(num_trials):
            seed = int(rng.integers(0, 1_000_000))
            instance = generate_airport_instance(
                num_items=num_items,
                max_cap=max_cap,
                max_budget=max_budget,
                num_players=num_players,
                rng_seed=seed,
            )

            start = perf_counter()
            result = run_airport_heuristic(instance, rng_seed=seed, **heuristic_kwargs)
            demands = result.get("demands", [])
            allocated_players = sum(1 for bundle in demands if len(bundle) > 0)
            allocation_rate = allocated_players / max(1, instance["num_players"])
            elapsed = perf_counter() - start

            history = result.get("meta", {}).get("history", [])
            if history:
                avg_change_rate = sum(h.get("demand_change_rate", 0.0) for h in history) / len(history)
                avg_max_price = sum(h.get("max_price", 0.0) for h in history) / len(history)
                avg_price_std = sum(h.get("price_std", 0.0) for h in history) / len(history)
            else:
                avg_change_rate = 0.0
                avg_max_price = 0.0
                avg_price_std = 0.0

            results.append(
                {
                    "num_items": num_items,
                    "max_cap": max_cap,
                    "num_players": num_players,
                    "trial": trial,
                    "slack_found": result.get("slack"),
                    "boosted": result.get("is_boosted", False),
                    "equilibrium": result.get("equilibrium", False),
                    "rounds": result.get("rounds"),
                    "delta_final": result.get("delta"),
                    "time_sec": elapsed,
                    "avg_demand_change": avg_change_rate,
                    "avg_max_price": avg_max_price,
                    "avg_price_std": avg_price_std,
                    "allocated_players": allocated_players,
                    "allocation_rate": allocation_rate,
                }
            )

df = pd.DataFrame(results)
df.head()


In [ ]:
# Summary metrics
summary = (
    df.groupby(["num_items", "max_cap", "num_players"])
    .agg(
        runs=("trial", "count"),
        equilibrium_rate=("equilibrium", "mean"),
        avg_slack=("slack_found", "mean"),
        boost_rate=("boosted", "mean"),
        avg_rounds=("rounds", "mean"),
        avg_time=("time_sec", "mean"),
        avg_demand_change=("avg_demand_change", "mean"),
        avg_allocated_players=("allocated_players", "mean"),
        avg_allocation_rate=("allocation_rate", "mean"),
    )
    .reset_index()
)
summary


In [ ]:
# Key aggregate numbers for paper text
print("rows:", len(df))
print("equilibrium_rate:", df["equilibrium"].mean())
print("avg_slack:", df["slack_found"].mean())
print("max_slack:", df["slack_found"].max())
print("boost_rate:", df["boosted"].mean())
print("avg_rounds:", df["rounds"].mean())
print("avg_runtime_sec:", df["time_sec"].mean())
print("avg_demand_change:", df["avg_demand_change"].mean())
print("avg_max_price:", df["avg_max_price"].mean())
print("avg_price_std:", df["avg_price_std"].mean())
print("avg_allocated_players:", df["allocated_players"].mean())
print("avg_allocation_rate:", df["allocation_rate"].mean())
print("\nslack_distribution:")
print(df["slack_found"].value_counts().sort_index())
print("\nmean_slack_by_max_cap:")
print(df.groupby("max_cap")["slack_found"].mean().sort_index())
print("\nmean_slack_by_num_players:")
print(df.groupby("num_players")["slack_found"].mean().sort_index())


In [ ]:
# Save results table
output_csv = "airport_simulation_results.csv"
df.to_csv(output_csv, index=False)
print(f"Saved results to {output_csv}")


In [ ]:
# Plot: Average Slack Needed vs Num Players
agg_slack = (
    df.groupby(["num_items", "max_cap", "num_players"])
    .agg(avg_slack=("slack_found", "mean"), std_slack=("slack_found", "std"))
    .reset_index()
)

num_items_levels = sorted(df["num_items"].unique())
max_cap_levels = sorted(df["max_cap"].unique())
color_map = {max_cap_levels[0]: "red", max_cap_levels[1]: "green", max_cap_levels[2]: "blue"}
marker_map = {num_items_levels[0]: "o", num_items_levels[1]: "s", num_items_levels[2]: "^"}
offsets = {max_cap_levels[0]: -1.2, max_cap_levels[1]: 0.0, max_cap_levels[2]: 1.2}

fig, ax = plt.subplots(figsize=(9, 5))
for num_items in num_items_levels:
    for max_cap in max_cap_levels:
        subset = agg_slack[(agg_slack["num_items"] == num_items) & (agg_slack["max_cap"] == max_cap)]
        if subset.empty:
            continue
        xs = subset["num_players"].to_numpy(dtype=float) + offsets[max_cap]
        ys = subset["avg_slack"].to_numpy(dtype=float)
        errs = subset["std_slack"].fillna(0.0).to_numpy(dtype=float)

        ax.errorbar(
            xs,
            ys,
            yerr=errs,
            fmt=marker_map[num_items],
            color=color_map[max_cap],
            alpha=0.65,
            markersize=8,
            markeredgecolor="black",
            markeredgewidth=0.5,
            linestyle="None",
            capsize=3,
        )

ax.set_xlabel("Number of players")
ax.set_ylabel("Average Slack Needed")
ax.set_title("Average Slack Needed vs Number of Players")
ax.grid(True, linestyle="--", alpha=0.3)

color_handles = [
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=color_map[c], label=f"max_cap={c}", markersize=8)
    for c in max_cap_levels
]
marker_handles = [
    plt.Line2D([0], [0], marker=marker_map[n], color="gray", linestyle="None", label=f"num_items={n}", markersize=8)
    for n in num_items_levels
]
ax.legend(handles=color_handles + marker_handles, loc="center left", bbox_to_anchor=(1.02, 0.5))

fig.tight_layout()
fig.savefig("airport_avg_slack_vs_num_players.png", dpi=200)
print("Saved airport_avg_slack_vs_num_players.png")


In [ ]:
# Plot: Max Slack Needed vs Num Players
agg_max_slack = (
    df.groupby(["num_items", "max_cap", "num_players"])
    .agg(max_slack=("slack_found", "max"))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 5))
for num_items in num_items_levels:
    for max_cap in max_cap_levels:
        subset = agg_max_slack[(agg_max_slack["num_items"] == num_items) & (agg_max_slack["max_cap"] == max_cap)]
        if subset.empty:
            continue
        xs = subset["num_players"].to_numpy(dtype=float) + offsets[max_cap]
        ys = subset["max_slack"].to_numpy(dtype=float)

        ax.scatter(
            xs,
            ys,
            c=color_map[max_cap],
            marker=marker_map[num_items],
            alpha=0.65,
            s=80,
            edgecolor="black",
            linewidth=0.5,
        )

ax.set_xlabel("Number of players")
ax.set_ylabel("Max Slack Needed")
ax.set_title("Max Slack Needed vs Number of Players")
ax.grid(True, linestyle="--", alpha=0.3)
ax.legend(handles=color_handles + marker_handles, loc="center left", bbox_to_anchor=(1.02, 0.5))

fig.tight_layout()
fig.savefig("airport_max_slack_vs_num_players.png", dpi=200)
print("Saved airport_max_slack_vs_num_players.png")


In [ ]:
# Plot: Average Demand Change vs Num Players
agg_change = (
    df.groupby(["num_items", "max_cap", "num_players"])
    .agg(avg_change=("avg_demand_change", "mean"), std_change=("avg_demand_change", "std"))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 5))
for num_items in num_items_levels:
    for max_cap in max_cap_levels:
        subset = agg_change[(agg_change["num_items"] == num_items) & (agg_change["max_cap"] == max_cap)]
        if subset.empty:
            continue
        xs = subset["num_players"].to_numpy(dtype=float) + offsets[max_cap]
        ys = subset["avg_change"].to_numpy(dtype=float)
        errs = subset["std_change"].fillna(0.0).to_numpy(dtype=float)

        ax.errorbar(
            xs,
            ys,
            yerr=errs,
            fmt=marker_map[num_items],
            color=color_map[max_cap],
            alpha=0.65,
            markersize=8,
            markeredgecolor="black",
            markeredgewidth=0.5,
            linestyle="None",
            capsize=3,
        )

ax.set_xlabel("Number of players")
ax.set_ylabel("Average Demand Change")
ax.set_title("Average Demand Change vs Number of Players")
ax.grid(True, linestyle="--", alpha=0.3)
ax.legend(handles=color_handles + marker_handles, loc="center left", bbox_to_anchor=(1.02, 0.5))

fig.tight_layout()
fig.savefig("airport_avg_demand_change_vs_num_players.png", dpi=200)
print("Saved airport_avg_demand_change_vs_num_players.png")


In [ ]:
# Additional diagnostics vs Num Players
perf = (
    df.groupby(["num_items", "max_cap", "num_players"])
    .agg(
        eq_rate=("equilibrium", "mean"),
        boost_rate=("boosted", "mean"),
        avg_rounds=("rounds", "mean"),
        std_rounds=("rounds", "std"),
        avg_time=("time_sec", "mean"),
        std_time=("time_sec", "std"),
        avg_max_price=("avg_max_price", "mean"),
        std_max_price=("avg_max_price", "std"),
        avg_price_std=("avg_price_std", "mean"),
        std_price_std=("avg_price_std", "std"),
    )
    .reset_index()
)

def _plot_airport_metric(metric, std_col, ylabel, title, filename):
    fig, ax = plt.subplots(figsize=(9, 5))
    for num_items in num_items_levels:
        for max_cap in max_cap_levels:
            subset = perf[(perf["num_items"] == num_items) & (perf["max_cap"] == max_cap)]
            if subset.empty:
                continue
            xs = subset["num_players"].to_numpy(dtype=float) + offsets[max_cap]
            ys = subset[metric].to_numpy(dtype=float)

            if std_col is None:
                ax.scatter(
                    xs, ys,
                    c=color_map[max_cap],
                    marker=marker_map[num_items],
                    alpha=0.65,
                    s=80,
                    edgecolor="black",
                    linewidth=0.5,
                )
            else:
                errs = subset[std_col].fillna(0.0).to_numpy(dtype=float)
                ax.errorbar(
                    xs, ys,
                    yerr=errs,
                    fmt=marker_map[num_items],
                    color=color_map[max_cap],
                    alpha=0.65,
                    markersize=8,
                    markeredgecolor="black",
                    markeredgewidth=0.5,
                    linestyle="None",
                    capsize=3,
                )

    ax.set_xlabel("Number of players")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend(handles=color_handles + marker_handles, loc="center left", bbox_to_anchor=(1.02, 0.5))
    fig.tight_layout()
    fig.savefig(filename, dpi=200)
    print(f"Saved {filename}")
    return fig, ax

_plot_airport_metric("eq_rate", None, "Equilibrium Rate", "Equilibrium Rate vs Number of Players", "airport_eq_rate_vs_num_players.png")
_plot_airport_metric("boost_rate", None, "Boost Usage Rate", "Boost Usage Rate vs Number of Players", "airport_boost_rate_vs_num_players.png")
_plot_airport_metric("avg_rounds", "std_rounds", "Average Rounds", "Average Rounds vs Number of Players", "airport_avg_rounds_vs_num_players.png")
_plot_airport_metric("avg_time", "std_time", "Average Runtime (s)", "Average Runtime vs Number of Players", "airport_avg_time_vs_num_players.png")
_plot_airport_metric("avg_max_price", "std_max_price", "Average Max Price", "Average Max Price vs Number of Players", "airport_avg_max_price_vs_num_players.png")
_plot_airport_metric("avg_price_std", "std_price_std", "Average Price Std Dev", "Average Price Std Dev vs Number of Players", "airport_avg_price_std_vs_num_players.png")


In [ ]:
# Plot: Players with a won pair vs Num Players
agg_alloc = (
    df.groupby(["num_items", "max_cap", "num_players"])
    .agg(
        avg_allocated_players=("allocated_players", "mean"),
        std_allocated_players=("allocated_players", "std"),
        avg_allocation_rate=("allocation_rate", "mean"),
        std_allocation_rate=("allocation_rate", "std"),
    )
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 5))
for num_items in num_items_levels:
    for max_cap in max_cap_levels:
        subset = agg_alloc[(agg_alloc["num_items"] == num_items) & (agg_alloc["max_cap"] == max_cap)]
        if subset.empty:
            continue
        xs = subset["num_players"].to_numpy(dtype=float) + offsets[max_cap]
        ys = subset["avg_allocated_players"].to_numpy(dtype=float)
        errs = subset["std_allocated_players"].fillna(0.0).to_numpy(dtype=float)

        ax.errorbar(
            xs,
            ys,
            yerr=errs,
            fmt=marker_map[num_items],
            color=color_map[max_cap],
            alpha=0.65,
            markersize=8,
            markeredgecolor="black",
            markeredgewidth=0.5,
            linestyle="None",
            capsize=3,
        )

ax.set_xlabel("Number of players")
ax.set_ylabel("Average players with a won pair")
ax.set_title("Players with a Won Pair vs Number of Players")
ax.grid(True, linestyle="--", alpha=0.3)
ax.legend(handles=color_handles + marker_handles, loc="center left", bbox_to_anchor=(1.02, 0.5))

fig.tight_layout()
fig.savefig("airport_allocated_players_vs_num_players.png", dpi=200)
print("Saved airport_allocated_players_vs_num_players.png")

# Optional normalised version (share of airlines served)
fig, ax = plt.subplots(figsize=(9, 5))
for num_items in num_items_levels:
    for max_cap in max_cap_levels:
        subset = agg_alloc[(agg_alloc["num_items"] == num_items) & (agg_alloc["max_cap"] == max_cap)]
        if subset.empty:
            continue
        xs = subset["num_players"].to_numpy(dtype=float) + offsets[max_cap]
        ys = subset["avg_allocation_rate"].to_numpy(dtype=float)
        errs = subset["std_allocation_rate"].fillna(0.0).to_numpy(dtype=float)

        ax.errorbar(
            xs,
            ys,
            yerr=errs,
            fmt=marker_map[num_items],
            color=color_map[max_cap],
            alpha=0.65,
            markersize=8,
            markeredgecolor="black",
            markeredgewidth=0.5,
            linestyle="None",
            capsize=3,
        )

ax.set_xlabel("Number of players")
ax.set_ylabel("Average allocation rate")
ax.set_title("Allocation Rate vs Number of Players")
ax.set_ylim(0.0, 1.05)
ax.grid(True, linestyle="--", alpha=0.3)
ax.legend(handles=color_handles + marker_handles, loc="center left", bbox_to_anchor=(1.02, 0.5))

fig.tight_layout()
fig.savefig("airport_allocation_rate_vs_num_players.png", dpi=200)
print("Saved airport_allocation_rate_vs_num_players.png")


In [ ]:
# Plot: Distribution of final perturbed capacities q + alpha* (no rerun)
# In this model, supply perturbation enters through slack alpha*, so the
# effective final capacity level is max_cap + slack_found.

cap_df = df.copy()
cap_df["effective_capacity"] = cap_df["max_cap"] + cap_df["slack_found"]

# Item-weighted distribution: each run contributes num_items slots at its
# effective final capacity level.
eff_item_counts = (
    cap_df.groupby("effective_capacity", as_index=False)["num_items"]
    .sum()
    .rename(columns={"num_items": "item_count"})
    .sort_values("effective_capacity")
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(
    eff_item_counts["effective_capacity"].to_numpy(dtype=float),
    eff_item_counts["item_count"].to_numpy(dtype=float),
    width=0.8,
    color="#4C78A8",
    edgecolor="black",
    linewidth=0.6,
    alpha=0.8,
)
ax.set_xlabel("Effective final capacity per slot (max_cap + alpha*)")
ax.set_ylabel("Number of slots across all runs")
ax.set_title("Distribution of Final Perturbed Capacities")
ax.grid(True, axis="y", linestyle="--", alpha=0.25)
fig.tight_layout()
fig.savefig("airport_final_perturbed_capacity_distribution.png", dpi=200)
print("Saved airport_final_perturbed_capacity_distribution.png")

# Optional split by baseline capacity level
fig, axes = plt.subplots(1, len(max_cap_levels), figsize=(4 * len(max_cap_levels), 4), sharey=True)
if len(max_cap_levels) == 1:
    axes = [axes]

for ax, cap in zip(axes, max_cap_levels):
    subset = cap_df[cap_df["max_cap"] == cap]
    item_counts = (
        subset.groupby("effective_capacity", as_index=False)["num_items"]
        .sum()
        .rename(columns={"num_items": "item_count"})
        .sort_values("effective_capacity")
    )
    ax.bar(
        item_counts["effective_capacity"].to_numpy(dtype=float),
        item_counts["item_count"].to_numpy(dtype=float),
        color=color_map.get(cap, "gray"),
        edgecolor="black",
        linewidth=0.6,
        alpha=0.75,
    )
    ax.set_title(f"max_cap={cap}")
    ax.set_xlabel("Effective capacity")
    ax.grid(True, axis="y", linestyle="--", alpha=0.25)

axes[0].set_ylabel("Number of slots")
fig.suptitle("Final Perturbed Capacity Distribution by Baseline max_cap", y=1.02)
fig.tight_layout()
fig.savefig("airport_final_perturbed_capacity_distribution_by_max_cap.png", dpi=200)
print("Saved airport_final_perturbed_capacity_distribution_by_max_cap.png")

eff_item_counts
